# 04 — Model comparison on `features_v1`
All four models run through the **same** OOF harness and the **same** folds, so their OOF/test probability matrices are row-aligned and directly blendable. We report OOF macro-F1 + per-class F1 for each and inspect feature importances.

In [1]:
# --- Shared toolbox (identical across notebooks; see build_notebooks.py) ---
import warnings, json
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, classification_report, confusion_matrix

SEED = 42
N_FOLDS = 5
CLASS_NAMES = {0: "Wake", 1: "Light", 2: "Deep", 3: "REM"}
CLASSES = np.array([0, 1, 2, 3])
EOG = "eog_burst_index"            # the only column with missing values (~50%)

RAW_FEATURES = [
    "eeg_delta_power", "eeg_theta_power", "eeg_alpha_power", "eeg_sigma_power",
    "eeg_beta_power", "eeg_gamma_power", "eeg_slow_osc_power", "eeg_spectral_entropy",
    "eeg_spindle_density", "eeg_kcomplex_rate", "emg_chin_tone", "emg_tone_variance",
    "eog_movement_density", "eog_amplitude", "heart_rate_mean", "heart_rate_variability",
    "respiration_rate", "respiration_variability", "spo2_mean", "body_movement_index",
    EOG,
]
POWER = ["eeg_delta_power", "eeg_theta_power", "eeg_alpha_power", "eeg_sigma_power",
         "eeg_beta_power", "eeg_gamma_power", "eeg_slow_osc_power"]

HERE = Path.cwd()
ART = HERE / "artifacts"; ART.mkdir(exist_ok=True)
SUB = HERE / "submissions"; SUB.mkdir(exist_ok=True)


def load_data():
    """Return (train_df, test_df). Features kept as-is (NaN preserved)."""
    tr = pd.read_csv(HERE / "train.csv")
    te = pd.read_csv(HERE / "test.csv")
    return tr, te


def macro_f1(y_true, y_pred):
    """Competition metric: macro-averaged F1 over the 4 classes."""
    return f1_score(y_true, y_pred, average="macro")


def per_class_f1(y_true, y_pred):
    f = f1_score(y_true, y_pred, average=None, labels=CLASSES)
    return {CLASS_NAMES[c]: round(float(f[i]), 4) for i, c in enumerate(CLASSES)}


def softplus(x):
    """Numerically stable log(1+exp(x)); strictly positive and monotonic.
    Used to turn z-scored band powers (~50% negative) into positive magnitudes
    so band ratios are well-defined instead of dividing by near-zero."""
    x = np.asarray(x, dtype=float)
    return np.log1p(np.exp(-np.abs(x))) + np.maximum(x, 0.0)


def _aligned_proba(model, X):
    """predict_proba with columns aligned to CLASSES = [0,1,2,3]."""
    p = model.predict_proba(X)
    cls = list(np.asarray(model.classes_))
    idx = [cls.index(c) for c in CLASSES]
    return p[:, idx]


def run_oof(make_model, X, y, X_test, folds, needs_impute=False, use_eval_set=False):
    """Out-of-fold training under fixed folds.

    Returns (oof, test_p, oof_macro, fold_scores):
      oof     : (n_train, 4) out-of-fold probabilities (each row predicted once)
      test_p  : (n_test, 4) test probabilities, averaged over the 5 fold-models
      oof_macro: global macro-F1 over the assembled OOF matrix (primary metric)

    Two model families, identical folds:
      - CatBoost (needs_impute=False): NaN passed through natively.
      - sklearn trees (needs_impute=True): add EOG-missing flag + fill EOG NaN
        with the TRAIN-FOLD median (fit on train fold only -> no leakage)."""
    n = len(y)
    oof = np.zeros((n, len(CLASSES)))
    test_p = np.zeros((len(X_test), len(CLASSES)))
    fold_scores = []
    for tr_idx, va_idx in folds:
        Xtr, Xva, Xte = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy(), X_test.copy()
        ytr, yva = y[tr_idx], y[va_idx]
        if needs_impute:
            med = Xtr[EOG].median()
            for d in (Xtr, Xva, Xte):
                if EOG + "_missing" not in d.columns:
                    d[EOG + "_missing"] = d[EOG].isna().astype("int8")
                d[EOG] = d[EOG].fillna(med)
            assert not Xtr.isna().any().any(), "NaN remained after impute"
        model = make_model()
        if use_eval_set:
            model.fit(Xtr, ytr, eval_set=(Xva, yva))
        else:
            model.fit(Xtr, ytr)
        oof[va_idx] = _aligned_proba(model, Xva)
        test_p += _aligned_proba(model, Xte) / len(folds)
        fold_scores.append(macro_f1(yva, oof[va_idx].argmax(1)))
    oof_macro = macro_f1(y, oof.argmax(1))
    return oof, test_p, oof_macro, fold_scores


def load_folds():
    """Load the fixed StratifiedKFold split saved by 02_baseline."""
    d = np.load(ART / "folds.npz", allow_pickle=True)
    return [(d[f"tr{i}"], d[f"va{i}"]) for i in range(N_FOLDS)]


In [2]:
def log_result(step, model, feature_set, oof_macro, pcf, notes=""):
    """Write one row to results_log.csv. Idempotent per (step, model): re-running
    a notebook replaces its own row rather than duplicating it."""
    import csv
    path = HERE / "results_log.csv"
    header = ["step", "model", "feature_set", "oof_macro_f1",
              "f1_Wake", "f1_Light", "f1_Deep", "f1_REM", "notes"]
    rows = []
    if path.exists():
        with open(path, newline="") as fh:
            data = list(csv.reader(fh))
        if data and data[0] == header:
            rows = [r for r in data[1:] if not (len(r) >= 2 and r[0] == step and r[1] == model)]
    row = [step, model, feature_set, round(float(oof_macro), 5),
           pcf["Wake"], pcf["Light"], pcf["Deep"], pcf["REM"], notes]
    with open(path, "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(header); w.writerows(rows); w.writerow(row)
    print("logged:", step, model, "OOF macro-F1 =", round(float(oof_macro), 5))


In [3]:
# Load the engineered feature matrices as DataFrames (NaN preserved)
cols = json.load(open(ART / "feature_cols.json"))["columns"]
Xtr = pd.DataFrame(np.load(ART / "features_v1_train.npy"), columns=cols)
Xte = pd.DataFrame(np.load(ART / "features_v1_test.npy"), columns=cols)
y = np.load(ART / "y_train.npy")
folds = load_folds()
print("features_v1:", Xtr.shape[1], "columns")

features_v1: 28 columns


In [4]:
from catboost import CatBoostClassifier
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              GradientBoostingClassifier)

def make_catboost():
    return CatBoostClassifier(loss_function="MultiClass",
        eval_metric="TotalF1:average=Macro", iterations=3000, learning_rate=0.03,
        depth=6, l2_leaf_reg=3.0, random_seed=SEED, od_type="Iter", od_wait=150,
        use_best_model=True, thread_count=-1, allow_writing_files=False, verbose=False)
def make_rf():
    return RandomForestClassifier(n_estimators=600, max_features="sqrt",
        min_samples_leaf=2, class_weight="balanced", n_jobs=-1, random_state=SEED)
def make_et():
    return ExtraTreesClassifier(n_estimators=800, max_features="sqrt",
        min_samples_leaf=1, class_weight="balanced", n_jobs=-1, random_state=SEED)
def make_gb():
    return GradientBoostingClassifier(n_estimators=400, learning_rate=0.05,
        max_depth=3, subsample=0.9, random_state=SEED)

specs = [("catboost", make_catboost, False, True),
         ("rf", make_rf, True, False),
         ("et", make_et, True, False),
         ("gb", make_gb, True, False)]
results = {}
for name, mk, imp, ev in specs:
    oof, tst, m, fs = run_oof(mk, Xtr, y, Xte, folds, needs_impute=imp, use_eval_set=ev)
    np.save(ART / f"{name}_oof.npy", oof)
    np.save(ART / f"{name}_test.npy", tst)
    results[name] = m
    print(f"{name:<10} OOF macro-F1 = {m:.5f} | per-class {per_class_f1(y, oof.argmax(1))}")
    log_result("04_models", name, "features_v1", m, per_class_f1(y, oof.argmax(1)))

catboost   OOF macro-F1 = 0.82898 | per-class {'Wake': 0.8516, 'Light': 0.8507, 'Deep': 0.7766, 'REM': 0.8369}
logged: 04_models catboost OOF macro-F1 = 0.82898


rf         OOF macro-F1 = 0.79203 | per-class {'Wake': 0.8195, 'Light': 0.8229, 'Deep': 0.7266, 'REM': 0.7992}
logged: 04_models rf OOF macro-F1 = 0.79203


et         OOF macro-F1 = 0.79462 | per-class {'Wake': 0.8284, 'Light': 0.8245, 'Deep': 0.7243, 'REM': 0.8012}
logged: 04_models et OOF macro-F1 = 0.79462


gb         OOF macro-F1 = 0.81284 | per-class {'Wake': 0.84, 'Light': 0.8377, 'Deep': 0.7598, 'REM': 0.8139}
logged: 04_models gb OOF macro-F1 = 0.81284


## Feature importances
Confirms the engineered features are actually used (evidence for the defense).

In [5]:
# Feature importance: a CatBoost fit on full data WITHOUT early stopping
# (use_best_model needs an eval_set; here we use fixed iterations instead).
cb_imp = CatBoostClassifier(loss_function="MultiClass", iterations=800,
    learning_rate=0.03, depth=6, l2_leaf_reg=3.0, random_seed=SEED,
    allow_writing_files=False, thread_count=-1, verbose=False)
cb_imp.fit(Xtr, y)
imp = cb_imp.get_feature_importance(prettified=True)
print("Top 15 CatBoost features:"); print(imp.head(15).to_string(index=False))

Top 15 CatBoost features:
             Feature Id  Importances
        eog_burst_index    11.655190
        eeg_sigma_power    10.335505
       respiration_rate     9.836645
      eeg_kcomplex_rate     6.610211
        eeg_alpha_power     5.724886
      theta_minus_alpha     5.330758
respiration_variability     4.625380
         eeg_beta_power     4.500429
   eog_movement_density     4.482911
          emg_chin_tone     3.412691
 heart_rate_variability     3.213937
    eeg_spindle_density     2.912435
              eeg_total     2.875824
        eeg_gamma_power     2.790104
          eog_amplitude     2.431647


Summary of OOF macro-F1 by model:

In [6]:
print({k: round(v, 5) for k, v in sorted(results.items(), key=lambda x: -x[1])})

{'catboost': 0.82898, 'gb': 0.81284, 'et': 0.79462, 'rf': 0.79203}
